In [ ]:
import re
import pandas as pd
from collections import Counter

df=pd.read_csv(r"C:/Users/yourname/Documents/DATASET.csv")

def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # remove anything in brackets (round, square, curly)
    text = re.sub(r'\(.*?\)|\[.*?\]|\{.*?\}', '', text)
    # remove "limited" or "ltd" (case insensitive, as whole words). Experimentation showed that the same companies often use limited & ltd haphazardly. This was a good fix.
    text = re.sub(r'\bltd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\blimited\b', '', text, flags=re.IGNORECASE)
    # remove all whitespace
    text = re.sub(r'\s+', '', text)
    return text.strip().lower()

# filter to relevant years. Relic from testing.
df_filtered = df[(df['Year'] >= 2015) & (df['Year'] <= 2024)].copy()

# apply normalization
df_filtered['B_norm'] = df_filtered['Company Name'].apply(normalize)

# build, for each year, a mapping of normalized name -> C value
year_maps = (
    df_filtered.groupby('Year')
    .apply(lambda g: dict(zip(g['B_norm'], g['License Class'])))
    .to_dict()
)

years = sorted(year_maps.keys())

transition_labels = ['A-B', 'A-C', 'B-A', 'B-C', 'C-A', 'C-B']
results = []

for y1, y2 in zip(years, years[1:]):
    map1 = year_maps[y1]
    map2 = year_maps[y2]
    common_names = set(map1.keys()) & set(map2.keys())

    counts = Counter()
    for name in common_names:
        c1, c2 = map1[name], map2[name]
        if c1 != c2:
            counts[f"{c1}-{c2}"] += 1

    row = {'Year_Pair': f"{y1}-{y2}"}
    for label in transition_labels:
        row[label] = counts.get(label, 0)
    results.append(row)

results_df = pd.DataFrame(results, columns=['Year_Pair'] + transition_labels)
print(results_df)

   Year_Pair  A-B  A-C  B-A  B-C  C-A  C-B
0  2015-2016    0    0   12    3    0    1
1  2016-2017    4    0   10    2    0    3
2  2017-2018    4    0    6    1    0    0
3  2018-2019    3    0    6    0    1    1
4  2019-2020    2    0   12    0    0    1
5  2020-2021    0    0   12    0    0    0
6  2021-2022    0    0    3    1    1    0
7  2022-2023    1    0    6    0    0    0
8  2023-2024    1    0    5    0    0    0


  .apply(lambda g: dict(zip(g['B_norm'], g['License Class'])))
